# 9-bob. 9-BOB. TARMOQ HUJUMLARINI ANIQLASH: SI ASOSIDAGI BOSTIRIB KIRISHNI ANIQLASH TIZIMLARI
## Amaliy misol: ma'lum va yangi hujumlarni aniqlash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/09-bob/9-2_malum-va-yangi-hujumlarni-aniqlash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 9-bob.*

Quyidagi dasturda sun'iy oqimlar to'plami yaratiladi. Har bir oqim beshta xususiyat bilan tavsiflanadi: davomiylik, chiquvchi va kiruvchi paketlar soni, o'rtacha paket hajmi va ulanishni boshlash bayrog'iga ega paketlar ulushi. To'plamda oddiy trafik va uchta ma'lum hujum turi bor: DoS, portlarni skanerlash va parolni tanlash. Ular asosida tasodifiy o'rmon modeli o'qitiladi. So'ngra modelga u hech qachon ko'rmagan to'rtinchi hujum turi, ya'ni ma'lumotlarni sizdirish ko'rsatiladi. Oxirida faqat oddiy trafikda o'qitilgan anomaliya detektori (3-bobda tanishgan izolyatsiya o'rmoni) bilan natijalar taqqoslanadi. Dastur amaliy mashg'ulotning A9 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

rng = np.random.default_rng(0)
ustunlar = ["davomiylik", "chiquvchi_paket", "kiruvchi_paket", "paket_hajmi", "syn_ulushi"]

def oqimlar(tur, n):
    """Har bir oqim uchun 5 ta xususiyat (sun'iy ma'lumot)."""
    if tur == "oddiy":
        d = [rng.lognormal(0.5, 1.4, n), rng.poisson(rng.lognormal(2.5, 0.8, n)),
             rng.poisson(rng.lognormal(2.7, 0.8, n)), rng.normal(650, 300, n), rng.beta(1, 10, n)]
    elif tur == "DoS":
        d = [rng.lognormal(-1, 0.8, n), rng.poisson(300, n), rng.poisson(5, n),
             rng.normal(120, 60, n), rng.beta(8, 3, n)]
    elif tur == "skanerlash":
        d = [rng.exponential(0.02, n), rng.poisson(1, n) + 1, rng.poisson(0.5, n),
             rng.normal(60, 5, n), rng.beta(12, 1, n)]
    elif tur == "parol_tanlash":
        d = [rng.lognormal(-0.3, 0.7, n), rng.poisson(9, n), rng.poisson(8, n),
             rng.normal(250, 120, n), rng.beta(1.5, 8, n)]
    elif tur == "sizdirish":   # o'qitishda ishtirok etmaydigan yangi hujum
        d = [rng.lognormal(2.6, 0.8, n), rng.poisson(rng.lognormal(4.3, 0.5, n)),
             rng.poisson(rng.lognormal(2.4, 0.6, n)), rng.normal(950, 250, n), rng.beta(1, 10, n)]
    X = pd.DataFrame(np.column_stack(d), columns=ustunlar).clip(lower=0)
    X["tur"] = tur
    return X

malum = pd.concat([oqimlar("oddiy", 8000), oqimlar("DoS", 700),
                   oqimlar("skanerlash", 700), oqimlar("parol_tanlash", 600)])
o, s = train_test_split(malum, test_size=0.3, stratify=malum.tur, random_state=0)

In [2]:
# 1. O'qituvchili model: faqat ma'lum hujumlar asosida o'qitiladi
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(o[ustunlar], o.tur)
print(classification_report(s.tur, model.predict(s[ustunlar]), digits=2))

               precision    recall  f1-score   support

          DoS       1.00      1.00      1.00       210
        oddiy       0.98      0.98      0.98      2400
parol_tanlash       0.77      0.76      0.76       180
   skanerlash       1.00      1.00      1.00       210

     accuracy                           0.97      3000
    macro avg       0.94      0.93      0.94      3000
 weighted avg       0.97      0.97      0.97      3000



In [3]:
# 2. Model hech qachon ko'rmagan hujum: ma'lumotlarni sizdirish
yangi = oqimlar("sizdirish", 300)
print("Yangi hujum oqimlari qaysi sinfga ajratildi:")
print(pd.Series(model.predict(yangi[ustunlar])).value_counts().to_string())

Yangi hujum oqimlari qaysi sinfga ajratildi:
oddiy    300


In [4]:
# 3. Anomaliyalarni aniqlash: faqat oddiy trafikda o'qitiladi
anomaliya = IsolationForest(contamination=0.02, random_state=0).fit(o[o.tur == "oddiy"][ustunlar])
oddiy_sinov = s[s.tur == "oddiy"][ustunlar]
print(f"\nAnomaliya detektori: yangi hujumning {np.mean(anomaliya.predict(yangi[ustunlar]) == -1):.0%} "
      f"qismini aniqladi, oddiy oqimlarning {np.mean(anomaliya.predict(oddiy_sinov) == -1):.1%} "
      f"qismini xato belgiladi")


Anomaliya detektori: yangi hujumning 57% qismini aniqladi, oddiy oqimlarning 1.7% qismini xato belgiladi


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
               precision    recall  f1-score   support

          DoS       1.00      1.00      1.00       210
        oddiy       0.98      0.98      0.98      2400
parol_tanlash       0.77      0.76      0.76       180
   skanerlash       1.00      1.00      1.00       210

     accuracy                           0.97      3000
    macro avg       0.94      0.93      0.94      3000
 weighted avg       0.97      0.97      0.97      3000

Yangi hujum oqimlari qaysi sinfga ajratildi:
oddiy    300

Anomaliya detektori: yangi hujumning 57% qismini aniqladi, oddiy oqimlarning 1.7% qismini xato belgiladi
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 9-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.